In [ ]:
# ============================================================
#  Full CPS Time-Series Engine with Feature Count
# ============================================================
import os, time, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from itertools import product
from sklearn.model_selection import TimeSeriesSplit
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import warnings
warnings.filterwarnings("ignore", message=".*pin_memory.*no accelerator.*")
# ============================================================
# Repro
# ============================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
# ============================================================
# Utils
# ============================================================
class Timer:
    def __enter__(self):
        self.start = time.time()
        return self
    def __exit__(self, *args):
        self.seconds = time.time() - self.start

def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)
# ============================================================
# Data helpers
# ============================================================
def robust_fill(df):
    df = df.ffill().bfill()
    return df.fillna(df.median(numeric_only=True))

def get_time_series_splits(X, n_splits=3, val_frac=0.15):
    n_samples = len(X)
    test_size = int(n_samples * val_frac)
    tscv = TimeSeriesSplit(n_splits=n_splits, test_size=test_size)
    return list(tscv.split(X))
# ============================================================
# Scaling / PCA
# ============================================================
class Standardizer:
    def fit(self, X):
        self.mean = X.mean(0, keepdims=True)
        self.std = X.std(0, keepdims=True)
        self.std[self.std < 1e-12] = 1.0
        return self
    def transform(self, X):
        return (X - self.mean) / self.std
class PCAz:
    def __init__(self, var=0.95):
        self.var = var
    def fit(self, X):
        Xc = X - X.mean(0)
        _, S, Vt = np.linalg.svd(Xc, full_matrices=False)
        ratio = (S**2) / (S**2).sum()
        k = np.searchsorted(np.cumsum(ratio), self.var) + 1
        self.mean_ = X.mean(0)
        self.comp_ = Vt[:k]
        return self
    def transform(self, X):
        return (X - self.mean_) @ self.comp_.T
# ============================================================
# Pipelines
# ============================================================
def add_y_lags(df, targets, lags):
    out = df.copy()
    for t in targets:
        for L in lags:
            out[f"{t}_lag{L}"] = out[t].shift(L)
    return out

def add_ewd(df, targets, spans):
    out = df.copy()
    for t in targets:
        for s in spans:
            ema = out[t].shift(1).ewm(span=s, adjust=False).mean()
            out[f"{t}_ewd{s}"] = ema
            out[f"{t}_res{s}"] = out[t] - ema
    return out

def build_pipeline(df, targets, name):
    if name == "BASE":
        X, Y = df.drop(columns=targets), df[targets]
    elif name == "Y_LAGS":
        tmp = add_y_lags(df, targets, [1])
        X, Y = tmp.drop(columns=targets), tmp[targets]
    elif name == "EWMA":
        tmp = add_ewd(df, targets, [8,16,32])
        X, Y = tmp.drop(columns=targets), tmp[targets]
    elif name == "PCA":
        X, Y = df.drop(columns=targets), df[targets]
    else:
        raise ValueError(name)
    X = X.select_dtypes(include=[np.number])
    mask = X.notna().all(axis=1)
    return X.loc[mask], Y.loc[mask]
# ============================================================
# Sequences
# ============================================================
def build_sequences(X, Y, window):
    X = np.asarray(X, dtype=np.float32)
    Y = np.asarray(Y, dtype=np.float32)
    X_seq = np.lib.stride_tricks.sliding_window_view(X, window_shape=(window, X.shape[1]))
    X_seq = X_seq.squeeze(axis=1)
    X_seq = X_seq.reshape(-1, window, X.shape[1])
    Y_seq = Y[window-1:]
    return X_seq, Y_seq
class SeqDataset(Dataset):
    def __init__(self, X, Y):
        self.X = torch.tensor(X)
        self.Y = torch.tensor(Y)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, i):
        return self.X[i], self.Y[i]

def make_loader(X, Y, batch):
    return DataLoader( SeqDataset(X, Y),batch_size=batch,shuffle=False,num_workers=0,pin_memory=True )
# ============================================================
# Models
# ============================================================
class RNNBase(nn.Module):
    def __init__(self, cell, in_dim, out_dim, hidden, layers, dropout):
        super().__init__()
        rnn = {"RNN": nn.RNN, "LSTM": nn.LSTM, "GRU": nn.GRU}[cell]
        self.cell = cell
        self.rnn = rnn(in_dim, hidden, layers, batch_first=True,
                       dropout=dropout if layers > 1 else 0.0)
        self.fc = nn.Linear(hidden, out_dim)
    def forward(self, x):
        out, h = self.rnn(x)
        h_last = h[0][-1] if self.cell == "LSTM" else h[-1]
        return self.fc(h_last)
class TCNBlock(nn.Module):
    def __init__(self, in_c, out_c, k, d):
        super().__init__()
        p = (k - 1) * d
        self.net = nn.Sequential(
            nn.Conv1d(in_c, out_c, k, padding=p, dilation=d),
            nn.ReLU(),
            nn.Dropout(dropout), 
            nn.Conv1d(out_c, out_c, k, padding=p, dilation=d),
            nn.ReLU())
        self.down = nn.Conv1d(in_c, out_c, 1) if in_c != out_c else nn.Identity()
    def forward(self, x):
        y = self.net(x)
        y = y[..., :x.size(-1)]
        return y + self.down(x)
class TCNReg(nn.Module):
    def __init__(self, in_dim, out_dim, hidden, layers, kernel):
        super().__init__()
        blocks, c, d = [], in_dim, 1
        for _ in range(layers):
            blocks.append(TCNBlock(c, hidden, kernel, d, dropout))
            c, d = hidden, d * 2
        self.tcn = nn.Sequential(*blocks)
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Linear(hidden, out_dim))
    def forward(self, x):
        return self.head(self.tcn(x.transpose(1, 2)))
class TransformerReg(nn.Module):
    def __init__(self, in_dim, out_dim, d_model, layers, heads,dropout=0.1):
        super().__init__()
        self.proj = nn.Linear(in_dim, d_model)
        enc = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=heads,
            dim_feedforward=2*d_model,dropout=dropout,batch_first=True)
        self.enc = nn.TransformerEncoder(enc, layers)
        self.fc = nn.Linear(d_model, out_dim)
    def forward(self, x):
        return self.fc(self.enc(self.proj(x)).mean(1))
# ============================================================
# Training / Prediction
# ============================================================
def convergence_epoch(val_curve, best_val, tol=0.05):
    threshold = best_val * (1 + tol)
    for i, v in enumerate(val_curve):
        if v <= threshold:
            return i + 1  # epochs are 1-based
    return len(val_curve)
def train_one(model,tr,va,epochs,lr,device,patience=5,val_every=2,):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()

    best = float("inf")
    best_state = None
    best_epoch = -1
    wait = 0

    train_curve = []
    val_curve = []
    # ---------- training loop ----------
    for epoch in range(1, epochs + 1):
        model.train()
        tr_losses = []

        for xb, yb in tr:
            xb = xb.to(device)
            yb = yb.to(device)

            opt.zero_grad()
            yhat = model(xb)
            loss = loss_fn(yhat, yb)
            loss.backward()
            opt.step()

            tr_losses.append(loss.item())

        train_curve.append(np.mean(tr_losses))
        # ---------- validation loss (for early stopping) ----------
        model.eval()
        if epoch % val_every == 0:
            with torch.no_grad():
                v = np.mean([loss_fn(model(xb.to(device)), yb.to(device)).item()
                    for xb, yb in va])
        else:
            v = best + 1e-8

        val_curve.append(v)

        if v < best:
            best = v
            best_epoch = epoch
            wait = 0
            best_state = {k: v.cpu() for k, v in model.state_dict().items()}
        else:
            wait += 1
            if wait >= patience:
                break
    # ---------- restore best model ----------
    model.load_state_dict(best_state)
    model.eval()
    # ---------- final per-target metrics ----------
    ys = []
    preds = []

    with torch.no_grad():
        for xb, yb in va:
            xb = xb.to(device)
            yb = yb.to(device)
            yhat = model(xb)

            ys.append(yb.cpu().numpy())
            preds.append(yhat.cpu().numpy())

    y_true = np.concatenate(ys, axis=0)   # (N, T)
    y_pred = np.concatenate(preds, axis=0)

    err = y_pred - y_true
    # per-target metrics
    val_mse_per_target = np.mean(err ** 2, axis=0)
    val_mae_per_target = np.mean(np.abs(err), axis=0)

    y_mean = np.mean(y_true, axis=0, keepdims=True)
    ss_res = np.sum((y_true - y_pred) ** 2, axis=0)
    ss_tot = np.sum((y_true - y_mean) ** 2, axis=0)

    val_r2_per_target = np.where(ss_tot > 0,1.0 - ss_res / ss_tot,np.nan,)
    # macro averages
    val_mse = np.mean(val_mse_per_target)
    val_mae = np.mean(val_mae_per_target)
    val_r2 = np.nanmean(val_r2_per_target)
    conv_ep = convergence_epoch(val_curve, best)
    return {"best_val_mse": best,
            "best_epoch": best_epoch,
            "stop_epoch": epoch,
            "convergence_epoch": conv_ep,
            "train_curve": train_curve,
            "val_curve": val_curve,
             # per-target
            "val_mse_per_target": val_mse_per_target,
            "val_mae_per_target": val_mae_per_target,
            "val_r2_per_target": val_r2_per_target,
             # macro
            "val_mse": val_mse,
            "val_mae": val_mae,
            "val_r2": val_r2,}

def predict(model, loader, device):
    model.eval()
    P, T = [], []
    with torch.no_grad():
        for xb, yb in loader:
            P.append(model(xb.to(device)).cpu().numpy())
            T.append(yb.numpy())
    return np.vstack(P), np.vstack(T)
# ============================================================
# Plotting
# ============================================================
def save_curves(train_curve, val_curve, out_dir, tag):
    """
    Saves publication-quality training/validation loss curves.
    Parameters:
        train_curve (list or np.array): Training loss per epoch.
        val_curve (list or np.array): Validation loss per epoch.
        out_dir (str): Directory to save the plot.
        tag (str): Descriptive tag for filename and plot title.
    """
    import matplotlib.pyplot as plt
    import os
    os.makedirs(out_dir, exist_ok=True)
    epochs = range(1, len(train_curve) + 1)

    plt.figure(figsize=(7, 5))
    plt.plot(epochs, train_curve, label="Train", color='tab:blue', linewidth=2, marker='o')
    plt.plot(epochs, val_curve, label="Val", color='tab:orange', linewidth=2, linestyle='--', marker='s')
    plt.xlabel("Epoch", fontsize=12)
    plt.ylabel("MSE", fontsize=12)
   # plt.title(f"Training & Validation Loss: {tag}", fontsize=14)
    plt.xticks(fontsize=10)
    plt.yticks(fontsize=10)
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.legend(fontsize=10)
    plt.tight_layout()
    # Save high-resolution PNG and PDF
    plt.savefig(os.path.join(out_dir, f"{tag}_curves.png"), dpi=300)
    plt.savefig(os.path.join(out_dir, f"{tag}_curves.pdf"), dpi=300)
    plt.close()

def plot_test(y, p, targets, path, zoom_windows=None, auto_zoom=True, zoom_window_size=100, top_k=3):
    """
    Publication-ready plotting for time series predictions.
    Features:
      - Consistent y-axis across targets
      - Full-range plot with shaded high-error regions
      - Optional zoomed-in views of top-error regions
      - High-res output for publications (PNG + PDF)
    """
    import matplotlib.patches as patches

    n_targets = len(targets)
    y_min = min(y.min(), p.min())
    y_max = max(y.max(), p.max())
    # Compute errors if auto-zoom
    if auto_zoom and zoom_windows is None:
        err = np.abs(y - p).mean(axis=1)
        top_indices = err.argsort()[-top_k:][::-1]
        zoom_windows = [(max(0, idx - zoom_window_size//2),
                         min(len(y), idx + zoom_window_size//2))
                        for idx in top_indices]
    # --- Full-range plot ---
    fig, axes = plt.subplots(n_targets, 1, figsize=(14, 3*n_targets), dpi=300)
    if n_targets == 1:
        axes = [axes]

    for i, t in enumerate(targets):
        axes[i].plot(y[:, i], label="Actual", color='tab:blue', linewidth=1.5)
        axes[i].plot(p[:, i], label="Pred", color='tab:orange', alpha=0.8, linewidth=1.5)
        axes[i].set_ylabel("Buffer Occupancy Level")
        axes[i].set_xlabel("Time Step")
        axes[i].set_ylim(y_min, y_max)
        axes[i].grid(True, linestyle="--", alpha=0.5)
        axes[i].legend(loc='upper left', fontsize=10)
        # Shade high-error regions
        if zoom_windows:
            for start, end in zoom_windows:
                axes[i].add_patch(patches.Rectangle((start, y_min), end-start, y_max-y_min,color='red', alpha=0.1, zorder=0))
    plt.tight_layout()
    # Save both PNG and PDF
    plt.savefig(path, dpi=300)
    plt.savefig(path.replace(".png", ".pdf"), dpi=300, format='pdf')  # ADD PDF
    plt.close()
    
    # --- Zoomed-in plots ---
    if zoom_windows:
        for start, end in zoom_windows:
            fig, axes = plt.subplots(n_targets, 1, figsize=(14, 3*n_targets), dpi=300)
            if n_targets == 1:
                axes = [axes]

            for i, t in enumerate(targets):
                axes[i].plot(y[start:end, i], label="Actual", color='tab:blue', linewidth=1.5)
                axes[i].plot(p[start:end, i], label="Pred", color='tab:orange', alpha=0.8, linewidth=1.5)
                axes[i].set_ylabel("Buffer Occupancy Level")
                axes[i].set_xlabel("Time Step")
                axes[i].set_ylim(y_min, y_max)
                axes[i].grid(True, linestyle="--", alpha=0.5)
                axes[i].legend(loc='upper left', fontsize=10)

            plt.tight_layout()
            zoom_png = path.replace(".png", f"_zoom_{start}_{end}.png")
            zoom_pdf = path.replace(".png", f"_zoom_{start}_{end}.pdf")  # ADD PDF PATH
            plt.savefig(zoom_png, dpi=300)
            plt.savefig(zoom_pdf, dpi=300, format='pdf')  # ADD PDF SAVE
            plt.close()
# ============================================================
# Metrics
# ============================================================
def r2_score(y_true, y_pred):
    ss_res = ((y_true - y_pred)**2).sum(axis=0)
    ss_tot = ((y_true - y_true.mean(axis=0))**2).sum(axis=0)
    return 1.0 - ss_res / (ss_tot + 1e-12)
def per_target_mse(y_true, y_pred):
    return ((y_true - y_pred)**2).mean(axis=0)
def per_target_mae(y_true, y_pred):
    return np.abs(y_true - y_pred).mean(axis=0)
def macro_mean(x):
    return float(np.mean(x))
def weighted_macro(x, weights):
    w = weights / (weights.sum() + 1e-12)
    return float((x * w).sum())
def rolling_metrics(y_true, y_pred, window=50, step=5):
    r2_list, mse_list = [], []
    for i in range(window, len(y_true), step):
        yt = y_true[i-window:i]
        yp = y_pred[i-window:i]
        r2_list.append(macro_mean(r2_score(yt, yp)))
        mse_list.append(macro_mean(per_target_mse(yt, yp)))
    return np.array(r2_list), np.array(mse_list)
# ============================================================
# MAIN
# ============================================================
def main():
    GLOBAL_BEST_VAL = float("inf")
    SEQ_CACHE = {}
    PCA_CACHE = {}
    set_seed(42)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    CSV = "/kaggle/input/datasets/musabadwan/final-cps-timeseries-features/Final_cps_timeseries_features.csv"
    TARGETS = ["Buffer_M1_M2", "Buffer_M2_M3"]
    PIPELINES = ["BASE","PCA","EWMA","Y_LAGS"] #"BASE","PCA",,"EWMA"
    MODELS = ["RNN","LSTM"]  # "GRU","RNN","TRANS"
    OUT = "./artifacts_full"
    os.makedirs(OUT, exist_ok=True)

    windows = [32,48,64]
    batches = [32,64]
    hiddens = [160,64,256]
    layers = [2,1]
    dropouts = [0.1,0.2,0.3]
    lrs = [1e-3,5e-4]
    GRID = [dict(window=w, batch=b, hidden=h, layers=l, lr=lr, dropout=d)
        for w, b, h, l, lr, d in product(windows, batches, hiddens, layers, lrs, dropouts)]
    df = robust_fill(pd.read_csv(CSV))
    X_values = df.drop(columns=TARGETS).select_dtypes(include=[np.number]).values
    Y_values = df[TARGETS].values

    # ----------------- TRUE HOLD-OUT SPLIT -----------------
    TEST_FRAC = 0.15
    N = len(df)
    test_size = int(N * TEST_FRAC)
    train_val_idx = np.arange(N - test_size)
    test_idx = np.arange(N - test_size, N)
    X_train_val = X_values[train_val_idx]
    Y_train_val = Y_values[train_val_idx]
    X_test = X_values[test_idx]
    Y_test = Y_values[test_idx]
    # ----------------- GRID SEARCH WITH CV -----------------
    selection_log = []
    for pipe, model_name, cfg in product(PIPELINES, MODELS, GRID):
        vals, times = [], []
        Xdf, Ydf = build_pipeline(df.iloc[train_val_idx], TARGETS, pipe)
        X_train_val = Xdf.values
        Y_train_val = Ydf.values
        splits = get_time_series_splits(X_train_val, n_splits=3, val_frac=0.15)

        for tr_idx, va_idx in splits:
            Xtr, Xva = X_train_val[tr_idx], X_train_val[va_idx]
            Ytr, Yva = Y_train_val[tr_idx], Y_train_val[va_idx]

            if pipe == "PCA":
                pca_key = (pipe, tuple(tr_idx))
                if pca_key not in PCA_CACHE:
                    PCA_CACHE[pca_key] = PCAz().fit(Xtr)
                pca = PCA_CACHE[pca_key]
                Xtr = pca.transform(Xtr)
                Xva = pca.transform(Xva)

            sc = Standardizer().fit(Xtr)
            Xtr, Xva = sc.transform(Xtr), sc.transform(Xva)

            key_tr = (pipe, cfg["window"], "train", tuple(tr_idx), Xtr.shape[1])
            key_va = (pipe, cfg["window"], "val", tuple(tr_idx), Xva.shape[1])
            if key_tr not in SEQ_CACHE:
                SEQ_CACHE[key_tr] = build_sequences(Xtr, Ytr, cfg["window"])
                SEQ_CACHE[key_va] = build_sequences(Xva, Yva, cfg["window"])

            Xtr_s, Ytr_s = SEQ_CACHE[key_tr]
            Xva_s, Yva_s = SEQ_CACHE[key_va]

            trL = make_loader(Xtr_s, Ytr_s, cfg["batch"])
            vaL = make_loader(Xva_s, Yva_s, cfg["batch"])

            n_features = Xtr_s.shape[2]

            if model_name in ["RNN", "LSTM", "GRU"]:
                model = RNNBase(model_name, n_features,len(TARGETS), cfg["hidden"], cfg["layers"],cfg["dropout"])
            else:
                model = TransformerReg(n_features, len(TARGETS), cfg["hidden"], cfg["layers"],4,cfg["dropout"])

            model.to(device)
            with Timer() as tt:
                info = train_one(model, trL, vaL, epochs=80, lr=cfg["lr"], device=device, patience=5, val_every=2)

            n_params = count_params(model)
            GLOBAL_BEST_VAL = min(GLOBAL_BEST_VAL, info["best_val_mse"])

            row = {"pipeline": pipe, "model": model_name,
                   "window": cfg["window"], "batch": cfg["batch"],
                   "hidden": cfg["hidden"], "layers": cfg["layers"],
                   "dropout": cfg["dropout"], "lr": cfg["lr"],
                   "best_val_mse": info["best_val_mse"], "convergence_epoch": info["convergence_epoch"],
                   "stop_epoch": info["stop_epoch"], "train_time_sec": tt.seconds,
                   "params": n_params, "n_features": n_features}

            for i, t in enumerate(TARGETS):
                row[f"cv_mse_{t}"] = info["val_mse_per_target"][i]
                row[f"cv_mae_{t}"] = info["val_mae_per_target"][i]
                row[f"cv_r2_{t}"] = info["val_r2_per_target"][i]

            row["cv_mse_macro"] = info["val_mse"]
            row["cv_mae_macro"] = info["val_mae"]
            row["cv_r2_macro"] = info["val_r2"]

            selection_log.append(row)

        print(f"[TRAINING] Pipeline={pipe}, Model={model_name}, Window={cfg['window']}, "
              f"Batch={cfg['batch']}, Layers={cfg['layers']}, Hidden={cfg['hidden']}, LR={cfg['lr']}, "
              f"Dropout={cfg['dropout']}, BestVal={info['best_val_mse']:.4f}, "
              f"BestEpoch={info['best_epoch']}, StopEpoch={info['stop_epoch']}, TimeTaken={tt.seconds:.2f}s")

    sel_df = pd.DataFrame(selection_log)
    sel_df.to_csv(f"{OUT}/grid_search_results.csv", index=False)
    # ----------------- AGGREGATE GRID SEARCH -----------------
    GROUP_COLS = ["pipeline", "model", "window", "batch", "hidden", "layers", "lr", "dropout"]
    agg = {}
    # ---- macro metrics ----
    for m in ["best_val_mse", "convergence_epoch", "train_time_sec"]:
        agg[m] = ["mean", "std"]
    # ---- per-target CV metrics ----
    for t in TARGETS:
        for m in ["cv_mse", "cv_mae", "cv_r2"]:
            agg[f"{m}_{t}"] = ["mean", "std"]
    # ---- macro CV metrics ----
    for m in ["cv_mse_macro", "cv_mae_macro", "cv_r2_macro"]:
        agg[m] = ["mean", "std"]
    # ---- static quantities ----
    agg["params"] = "first"
    agg["n_features"] = "first"
    
    grid_df = sel_df.groupby(GROUP_COLS).agg(agg)
    
    # flatten MultiIndex columns
    grid_df.columns = [f"{c[0]}_{c[1]}" if isinstance(c, tuple) else c for c in grid_df.columns]
    
    grid_df = grid_df.reset_index()
    grid_df.to_csv(f"{OUT}/grid_search_results.csv", index=False)
    # ----------------- FINAL TEST -----------------
    results = []

    for (pipe, model_name), g in sel_df.groupby(["pipeline", "model"]):
        best = g.sort_values("best_val_mse").iloc[0]
        cfg = {"window": int(best["window"]),
               "batch": int(best["batch"]),
               "hidden": int(best["hidden"]),
               "layers": int(best["layers"]),
               "dropout": float(best["dropout"]),
               "lr": float(best["lr"])}

        # Build full train_val + test sequences
        Xdf_train_val, Ydf_train_val = build_pipeline(df.iloc[train_val_idx], TARGETS, pipe)
        Xdf_test, Ydf_test = build_pipeline(df.iloc[test_idx], TARGETS, pipe)

        Xtr, Xte = Xdf_train_val.values, Xdf_test.values
        Ytr, Yte = Ydf_train_val.values, Ydf_test.values

        if pipe == "PCA":
            pca = PCAz().fit(Xtr)
            Xtr = pca.transform(Xtr)
            Xte = pca.transform(Xte)

        sc = Standardizer().fit(Xtr)
        Xtr_s, Ytr_s = build_sequences(sc.transform(Xtr), Ytr, cfg["window"])
        Xte_s, Yte_s = build_sequences(sc.transform(Xte), Yte, cfg["window"])
        # Save test indices for alignment
        test_idx_seq = test_idx[cfg["window"] - 1:]  # align sequences
        model_out_dir = os.path.join(OUT, f"{pipe}_{model_name}")
        os.makedirs(model_out_dir, exist_ok=True)
        np.save(os.path.join(model_out_dir, "test_indices.npy"), test_idx_seq)

        trL = make_loader(Xtr_s, Ytr_s, cfg["batch"])
        teL = make_loader(Xte_s, Yte_s, cfg["batch"])

        n_features = Xtr_s.shape[2]
        if model_name in ["RNN", "LSTM", "GRU"]:
            model = RNNBase(model_name, n_features, len(TARGETS), cfg["hidden"], cfg["layers"], cfg["dropout"])
        else:
            model = TransformerReg(n_features, len(TARGETS), cfg["hidden"], cfg["layers"], 4, cfg["dropout"])

        model.to(device)
        with Timer() as tt:
            info=train_one(model, trL, trL, 80, cfg["lr"], device,patience=5,val_every=2)
        # Save the final training curve plot
        save_curves(info["train_curve"], info["val_curve"], OUT, f"{pipe}_{model_name}_train_curve")
        P, T = predict(model, teL, device)
        # Save predictions and true values aligned with test indices
        np.save(os.path.join(OUT, f"{pipe}_{model_name}_pred.npy"), P)
        np.save(os.path.join(OUT, f"{pipe}_{model_name}_true.npy"), T)
        mse_t = per_target_mse(T, P)
        mae_t = per_target_mae(T, P)
        r2_t = r2_score(T, P)

        row = {"pipeline": pipe, "model": model_name, "n_features": n_features,
               "mse_macro": macro_mean(mse_t), "mae_macro": macro_mean(mae_t), "r2_macro": macro_mean(r2_t),
               "mse_weighted": weighted_macro(mse_t, T.var(axis=0)), "r2_weighted": weighted_macro(r2_t, T.var(axis=0)),
               "window": cfg["window"], "batch": cfg["batch"],
               "hidden": cfg["hidden"], "layers": cfg["layers"], "dropout": cfg["dropout"], "lr": cfg["lr"],
               "params": count_params(model),"train_time_sec": tt.seconds,"convergence_epoch": info["convergence_epoch"]}

        for i, t in enumerate(TARGETS):
            row[f"mse_{t}"] = mse_t[i]
            row[f"mae_{t}"] = mae_t[i]
            row[f"r2_{t}"] = r2_t[i]

        results.append(row)
        plot_test(y=T, p=P, targets=TARGETS, path=f"{OUT}/{pipe}_{model_name}_test.png")
        metrics_path = os.path.join(OUT, f"{pipe}_{model_name}_metrics.csv")
        metrics_df = pd.DataFrame([row])
        metrics_df.to_csv(metrics_path, index=False)
    results_df = pd.DataFrame(results)
    results_df["rank_r2"] = results_df["r2_macro"].rank(ascending=False, method="min").astype(int)
    results_df.sort_values("rank_r2").to_csv(f"{OUT}/final_results_ranked.csv", index=False)
if __name__ == "__main__":
    main()


In [ ]:
import os
import zipfile

# Folder containing all your output files
output_folder = '/kaggle/working/'  # or '/kaggle/output/'

# Name of the zip file you want to create
zip_filename = '/kaggle/working/all_outputs.zip'

# Create a zip file
with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk(output_folder):
        for file in files:
            # Avoid zipping the zip itself if re-running
            if file != os.path.basename(zip_filename):
                file_path = os.path.join(root, file)
                # Preserve folder structure relative to output_folder
                arcname = os.path.relpath(file_path, output_folder)
                zipf.write(file_path, arcname)

print(f"All files zipped into {zip_filename}")
